
# 06 — Model Evaluation

## Objective

Evaluate the trained CNN on unseen test images.

We calculate:

- Test accuracy
- Precision per class
- Recall per class
- F1-score per class
- Confusion matrix

Accuracy alone is not sufficient to understand a
medical-image classifier.

Recall is especially important to inspect because
it shows how frequently examples of a particular
class are correctly identified.

The dataset may contain acquisition-related biases,
and we have not verified patient-level independence.
Therefore, test results describe performance on this
dataset rather than clinical diagnostic reliability.


In [2]:

from pathlib import Path
import sys

import pandas as pd
import tensorflow as tf

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.brainscan import create_dataset

# Load the best V3 model
model_path = (
    PROJECT_ROOT
    / "models"
    / "experiments"
    / "v3_scheduler"
    / "best_cnn.keras"
)

assert model_path.is_file(), f"Missing: {model_path}"

model = tf.keras.models.load_model(model_path)

print("Evaluating model:", model_path)
model.summary()



I0000 00:00:1792169598.260013  144968 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1792169598.305510  144968 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1792169599.217506  144968 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
W0000 00:00:1792169600.320397  144968 gpu_device.cc:2459] TensorFlow was not built with CUDA kernel binaries compatible with compute c

Evaluating model: /home/houssam/Desktop/Briefs/Brief 4/models/experiments/v3_scheduler/best_cnn.keras


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 112, 112, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 56, 56, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 28, 28, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling2d        │ (None, 128)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │        33,024 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 64)             │        16,448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 4)              │           260 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 872,782 (3.33 MB)

 Trainable params: 290,692 (1.11 MB)

 Non-trainable params: 704 (2.75 KB)

 Optimizer params: 581,386 (2.22 MB)

In [3]:

manifest = pd.read_csv(
    PROJECT_ROOT / "data/splits/split_manifest.csv"
)

def get_records(split_name):
    rows = manifest[manifest["split"] == split_name]

    return [
        (PROJECT_ROOT / path, label)
        for path, label in zip(
            rows["path"],
            rows["label"]
        )
    ]

val_records = get_records("validation")
test_records = get_records("test")

val_ds = create_dataset(
    val_records,
    training=False,
    batch_size=32
)

test_ds = create_dataset(
    test_records,
    training=False,
    batch_size=32
)

val_loss, val_accuracy = model.evaluate(
    val_ds,
    verbose=0
)

test_loss, test_accuracy = model.evaluate(
    test_ds,
    verbose=0
)

print(f"Validation accuracy: {val_accuracy:.2%}")
print(f"Validation loss: {val_loss:.4f}")

print(f"Test accuracy: {test_accuracy:.2%}")
print(f"Test loss: {test_loss:.4f}")

print(
    f"Accuracy gap: "
    f"{(val_accuracy - test_accuracy) * 100:.2f} pp"
)


I0000 00:00:1792169601.435583  144968 generator_dataset_op.cc:213] Memory patch applied: M_TRIM_THRESHOLD=128 kb was set.
I0000 00:00:1792169601.747125  154562 service.cc:153] XLA service 0x7c57ac05b330 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1792169601.747145  154562 service.cc:161]   StreamExecutor [0]: NVIDIA GeForce RTX 5070 Laptop GPU, Compute Capability 12.0a (Driver: 13.3.0; Runtime: 12.9.0; Toolkit: 12.5.0; DNN: 9.27.0)
I0000 00:00:1792169601.759599  154562 dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1792169601.834057  154562 cuda_dnn.cc:461] Loaded cuDNN version 92700
I0000 00:00:1792169603.817061  154562 device_compiler.h:208] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
/home/houssam/Desktop/Briefs/Brief 4/.venv-gpu/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:164: U

Validation accuracy: 92.96%
Validation loss: 0.2159
Test accuracy: 91.58%
Test loss: 0.2170
Accuracy gap: 1.39 pp


I0000 00:00:1792169608.720513  154637 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 6645825189600942948
I0000 00:00:1792169608.720543  154637 local_rendezvous.cc:432] Local rendezvous recv item cancelled. Key hash: 13491511308658363688


In [4]:

cm = confusion_matrix(
    y_true,
    y_pred,
    labels=list(range(len(CLASS_NAMES)))
)

display = ConfusionMatrixDisplay(
    confusion_matrix=cm,
    display_labels=CLASS_NAMES
)

display.plot(
    cmap="Blues",
    xticks_rotation=45
)

plt.title("BrainScan AI — Confusion Matrix")
plt.tight_layout()
plt.show()

report_path = PROJECT_ROOT / "models/test_report.txt"
report_path.write_text(
    f"Test accuracy: {test_accuracy:.4f}\n\n{report}"
)

print("Report saved:", report_path)


NameError: name 'confusion_matrix' is not defined